# Forest Fire Detection — Image Data Preprocessing
## Notebook 02: Data Loading, Cleaning, Splitting & Augmentation


In [ ]:
import os, sys, json, shutil, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split

ROOT      = Path(r"e:/Sharvayu data/Malware/Symbiosis Nagpur SIT/7th SEM/Forest Fire task")
IMPL      = ROOT / "Implementation"
PROC_DIR  = IMPL / "data" / "processed"
PLOTS_DIR = IMPL / "artifacts" / "plots"
META_DIR  = IMPL / "artifacts" / "metadata"
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Dataset paths
FIRE_DIR    = ROOT / "archive (1)" / "fire_dataset" / "fire_images"
NOFIRE_DIR  = ROOT / "archive (1)" / "fire_dataset" / "non_fire_images"

UAVS_RAW = ROOT / "forestfire-8gb" / "UAVS-FDDB UAVs-based Forest Fire Detection Database" / "Original Image Dataset (Raw Images)"
UAVS_FIRE_DIRS   = [UAVS_RAW / "Evening Fire Incident_raw_img" / "Evening Fire Incident_raw_img",
                     UAVS_RAW / "Pre-Evening Fire Incident_raw_img" / "Pre-Evening Fire Incident_raw_img"]
UAVS_NOFIRE_DIRS = [UAVS_RAW / "Evening Forest condition_raw_img" / "Evening Forest condition_raw_img",
                     UAVS_RAW / "Pre-evening Forest condition_raw_img" / "Pre-evening Forest condition_raw_img"]

IMAGE_SIZE = 224
RANDOM_SEED = 42

print("Image Preprocessing Pipeline")
print("="*60)
print(f"Target image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Random seed: {RANDOM_SEED}")


## 1. Collect All Image Paths

In [ ]:
def collect_images(directories, label, name=""):
    paths = []
    for d in directories:
        d = Path(d)
        if d.exists():
            for ext in ['*.png','*.jpg','*.jpeg','*.bmp','*.tiff']:
                paths.extend(list(d.glob(ext)))
            for ext in ['*.PNG','*.JPG','*.JPEG']:
                paths.extend(list(d.glob(ext)))
    print(f"  {name or label}: {len(paths)} images")
    return [(str(p), label) for p in paths]

# Archive dataset
arch_fire   = collect_images([FIRE_DIR],   'FIRE',    "Archive FIRE")
arch_nofire = collect_images([NOFIRE_DIR], 'NO_FIRE', "Archive NO_FIRE")

# UAVS dataset
uavs_fire   = collect_images(UAVS_FIRE_DIRS,   'FIRE',    "UAVS FIRE")
uavs_nofire = collect_images(UAVS_NOFIRE_DIRS, 'NO_FIRE', "UAVS NO_FIRE")

all_data = arch_fire + arch_nofire + uavs_fire + uavs_nofire
print(f"\nTotal before deduplication: {len(all_data)}")
print(f"  FIRE:    {sum(1 for _,l in all_data if l=='FIRE')}")
print(f"  NO_FIRE: {sum(1 for _,l in all_data if l=='NO_FIRE')}")


## 2. Deduplication — Remove Exact Duplicates

In [ ]:
import random
random.seed(RANDOM_SEED)

def file_hash(path, chunk_size=8192):
    h = hashlib.md5()
    try:
        with open(path, 'rb') as f:
            while chunk := f.read(chunk_size):
                h.update(chunk)
        return h.hexdigest()
    except:
        return None

print("Computing file hashes for deduplication...")
print("(Sampling for speed — checking all unique paths)")

seen_hashes = set()
unique_data = []
duplicates  = 0
corrupt     = 0

for i, (path, label) in enumerate(all_data):
    if i % 200 == 0:
        print(f"  Processed {i}/{len(all_data)}...", end='\r')
    h = file_hash(path)
    if h is None:
        corrupt += 1
        continue
    if h in seen_hashes:
        duplicates += 1
    else:
        seen_hashes.add(h)
        # Also validate image can be opened
        try:
            with Image.open(path) as im:
                im.verify()
            unique_data.append((path, label))
        except:
            corrupt += 1

print(f"\nDuplicates removed: {duplicates}")
print(f"Corrupt images:     {corrupt}")
print(f"Unique valid images: {len(unique_data)}")
print(f"  FIRE:    {sum(1 for _,l in unique_data if l=='FIRE')}")
print(f"  NO_FIRE: {sum(1 for _,l in unique_data if l=='NO_FIRE')}")


## 3. Train / Validation / Test Split (Stratified)

In [ ]:
paths  = [p for p,_ in unique_data]
labels = [l for _,l in unique_data]

# First split: 80% train+val, 20% test
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    paths, labels, test_size=0.20, random_state=RANDOM_SEED, stratify=labels
)

# Second split: 75% train, 25% val (of the 80%)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=0.25, random_state=RANDOM_SEED, stratify=train_val_labels
)

print("Dataset Split Results:")
print(f"  Training:   {len(train_paths)} images")
print(f"    FIRE:     {train_labels.count('FIRE')}")
print(f"    NO_FIRE:  {train_labels.count('NO_FIRE')}")
print(f"  Validation: {len(val_paths)} images")
print(f"    FIRE:     {val_labels.count('FIRE')}")
print(f"    NO_FIRE:  {val_labels.count('NO_FIRE')}")
print(f"  Test:       {len(test_paths)} images")
print(f"    FIRE:     {test_labels.count('FIRE')}")
print(f"    NO_FIRE:  {test_labels.count('NO_FIRE')}")

# Save split
split_data = {
    'train': list(zip(train_paths, train_labels)),
    'val':   list(zip(val_paths, val_labels)),
    'test':  list(zip(test_paths, test_labels))
}
with open(META_DIR / "data_splits.json", "w") as f:
    json.dump(split_data, f, indent=2)
print(f"\nSplit saved to: {META_DIR}/data_splits.json")


## 4. Class Imbalance Analysis & Weighting

In [ ]:
from collections import Counter

train_counter = Counter(train_labels)
total_train = len(train_labels)

print("Class distribution in training set:")
for cls, cnt in train_counter.items():
    print(f"  {cls}: {cnt} ({100*cnt/total_train:.1f}%)")

# Compute class weights for loss function
n_fire    = train_counter.get('FIRE', 1)
n_nofire  = train_counter.get('NO_FIRE', 1)
n_total   = total_train

# Sklearn-style balanced weights
weight_fire   = n_total / (2 * n_fire)
weight_nofire = n_total / (2 * n_nofire)

CLASS_NAMES = ['FIRE', 'NO_FIRE']
class_weights = {'FIRE': weight_fire, 'NO_FIRE': weight_nofire}

print(f"\nComputed class weights:")
for cls, w in class_weights.items():
    print(f"  {cls}: {w:.4f}")

# Save class weights
with open(META_DIR / "class_weights.json", "w") as f:
    json.dump({'class_names': CLASS_NAMES, 'class_weights': class_weights}, f, indent=2)


## 5. Data Augmentation Configuration

In [ ]:
# Define augmentation pipeline (torchvision transforms)
# These are applied only to TRAINING set
import torchvision.transforms as T
import torch

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = T.Compose([
    T.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    T.RandomCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    T.RandomGrayscale(p=0.02),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_test_transforms = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Save transform config
transform_config = {
    'image_size': IMAGE_SIZE,
    'imagenet_mean': IMAGENET_MEAN,
    'imagenet_std': IMAGENET_STD,
    'train_augmentations': [
        'Resize(256x256)', 'RandomCrop(224x224)', 'RandomHorizontalFlip(p=0.5)',
        'RandomVerticalFlip(p=0.2)', 'RandomRotation(10°)',
        'ColorJitter(brightness=0.2,contrast=0.2)', 'RandomGrayscale(p=0.02)',
        'Normalize(ImageNet mean/std)'
    ],
    'val_test_augmentations': ['Resize(224x224)', 'Normalize(ImageNet mean/std)']
}
with open(META_DIR / "transform_config.json", "w") as f:
    json.dump(transform_config, f, indent=2)

print("Augmentation Pipeline:")
print("  Training transforms:")
for aug in transform_config['train_augmentations']:
    print(f"    • {aug}")
print("  Val/Test transforms:")
for aug in transform_config['val_test_augmentations']:
    print(f"    • {aug}")


## 6. Visualize Augmented Samples

In [ ]:
import random
random.seed(42)

def show_augmented(img_path, n_aug=6):
    base_transform = T.Resize((IMAGE_SIZE, IMAGE_SIZE))
    aug_transform  = T.Compose([
        T.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
        T.RandomCrop(IMAGE_SIZE),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=10),
        T.ColorJitter(brightness=0.3, contrast=0.2),
    ])
    img_pil = Image.open(img_path).convert('RGB')
    imgs = [base_transform(img_pil)] + [aug_transform(img_pil) for _ in range(n_aug)]
    return imgs

# Pick one fire and one non-fire
fire_sample_path   = random.choice(train_paths[:50] if train_labels[0]=='FIRE' else
                                   [p for p,l in zip(train_paths,train_labels) if l=='FIRE'][:10])
nofire_sample_path = random.choice([p for p,l in zip(train_paths,train_labels) if l=='NO_FIRE'][:10])

fig, axes = plt.subplots(2, 7, figsize=(18, 6))
fig.suptitle("Original + Augmented Samples", fontsize=13, fontweight='bold')
titles = ['Original', 'Aug 1', 'Aug 2', 'Aug 3', 'Aug 4', 'Aug 5', 'Aug 6']

for row_idx, (img_path, row_label, row_color) in enumerate([
    (fire_sample_path, 'FIRE', 'red'),
    (nofire_sample_path, 'NO FIRE', 'green')
]):
    imgs = show_augmented(img_path)
    for col_idx, img in enumerate(imgs):
        axes[row_idx][col_idx].imshow(img)
        axes[row_idx][col_idx].axis('off')
        t = titles[col_idx] if col_idx > 0 else f"{row_label}\n{titles[0]}"
        axes[row_idx][col_idx].set_title(t, fontsize=8,
                                          color=row_color if col_idx==0 else 'black')

plt.tight_layout()
plt.savefig(PLOTS_DIR / "augmentation_samples.png", dpi=100, bbox_inches='tight')
plt.close()
print("Augmentation visualization saved.")


In [ ]:
print("\nPreprocessing Summary:")
print(f"  Total unique images: {len(unique_data)}")
print(f"  Training:   {len(train_paths)} ({100*len(train_paths)/len(unique_data):.0f}%)")
print(f"  Validation: {len(val_paths)} ({100*len(val_paths)/len(unique_data):.0f}%)")
print(f"  Test:       {len(test_paths)} ({100*len(test_paths)/len(unique_data):.0f}%)")
print(f"  Class weights: FIRE={weight_fire:.3f}, NO_FIRE={weight_nofire:.3f}")
print(f"  Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print("\nNotebook 02 complete. Ready for model benchmarking.")
